# Threat Hunting with ML

## What This Is
Threat hunting is the proactive search for threats that have evaded automated detection. ML-assisted hunting uses unsupervised methods to surface suspicious clusters of activity for analyst investigation.

## Hunting Workflow
1. **Data collection**: Aggregate endpoint telemetry, network flows, process trees
2. **Feature engineering**: Extract behavioral features per entity (host, user, process)
3. **Dimensionality reduction**: UMAP for visualization, preserves local structure
4. **Clustering**: HDBSCAN for density-based clustering — finds arbitrary-shaped clusters, handles noise
5. **Triage**: Analyst examines small clusters or outliers (noise points in HDBSCAN)

## Why UMAP + HDBSCAN
- **UMAP** preserves local topology better than t-SNE for large datasets and is faster
- **HDBSCAN** doesn't require specifying number of clusters; noise points are labeled -1 and are prime hunting candidates

In [ ]:
import numpy as np
from typing import List, Dict, Tuple, Optional

np.random.seed(42)

# Simulate endpoint telemetry features
# One row = one process execution context

def generate_normal_endpoint_events(n: int) -> np.ndarray:
    """Simulate normal process behavior features."""
    return np.column_stack([
        np.random.choice([0, 1, 2], n, p=[0.6, 0.3, 0.1]),  # process_type (0=user,1=sys,2=svc)
        np.random.normal(50, 20, n).clip(0, 100),             # cpu_pct
        np.random.exponential(100, n),                         # memory_mb
        np.random.poisson(5, n),                               # child_processes
        np.random.poisson(10, n),                              # file_ops
        np.random.poisson(3, n),                               # network_conns
        np.random.binomial(1, 0.05, n),                        # spawned_cmd
        np.random.binomial(1, 0.02, n),                        # modified_autorun
        np.random.normal(12, 4, n).clip(6, 22),               # hour_of_day
        np.random.exponential(50, n),                          # bytes_written
    ])

def generate_attack_events(attack_type: str, n: int) -> np.ndarray:
    """Simulate different attack pattern features."""
    base = generate_normal_endpoint_events(n)
    if attack_type == 'lateral_movement':
        base[:, 3] = np.random.poisson(50, n)   # many child processes
        base[:, 5] = np.random.poisson(30, n)   # many network connections
        base[:, 6] = np.ones(n)                  # always spawns cmd
    elif attack_type == 'ransomware':
        base[:, 4] = np.random.poisson(1000, n) # massive file ops
        base[:, 9] = np.random.exponential(50000, n)  # huge bytes written
        base[:, 2] = np.random.normal(2000, 500, n)   # high memory (crypto)
    elif attack_type == 'persistence':
        base[:, 7] = np.ones(n)                  # always modifies autorun
        base[:, 8] = np.random.uniform(0, 5, n)  # early morning
    return base

# Generate dataset
X_normal = generate_normal_endpoint_events(500)
X_lateral = generate_attack_events('lateral_movement', 30)
X_ransom = generate_attack_events('ransomware', 20)
X_persist = generate_attack_events('persistence', 25)

X = np.vstack([X_normal, X_lateral, X_ransom, X_persist])
true_labels = ['normal']*500 + ['lateral']*30 + ['ransomware']*20 + ['persistence']*25

print(f'Endpoint telemetry: {len(X)} events, {X.shape[1]} features')
print('Attack events hidden within normal traffic')


In [ ]:
# Simplified UMAP via PCA + neighbor graph
# In production: use umap-learn package

# Normalize features
mu, sig = X.mean(0), X.std(0) + 1
X_n = (X - mu) / sig

# PCA to 2D for visualization
X_c = X_n - X_n.mean(0)
cov = X_c.T @ X_c / len(X_c)
vals, vecs = np.linalg.eigh(cov)
top = vecs[:, np.argsort(vals)[-2:]]
X_2d = X_c @ top

# Simplified HDBSCAN: identify outliers via distance to k-th nearest neighbor
def knn_outlier_score(X: np.ndarray, k: int = 10) -> np.ndarray:
    """Score each point by its k-NN distance (higher = more outlier-like)."""
    scores = []
    for i, x in enumerate(X):
        dists = np.linalg.norm(X - x, axis=1)
        dists[i] = np.inf
        knn_dist = np.sort(dists)[:k].mean()
        scores.append(knn_dist)
    return np.array(scores)

# Sample for speed
sample_idx = np.random.choice(len(X_n), 200, replace=False)
X_sample = X_n[sample_idx]
labels_sample = [true_labels[i] for i in sample_idx]

outlier_scores = knn_outlier_score(X_sample, k=5)

# Flag top 10% as suspicious
threshold = np.percentile(outlier_scores, 90)
flagged = outlier_scores > threshold

# How many attack events were in top 10% outliers?
n_attacks = sum(1 for l in labels_sample if l != 'normal')
flagged_attacks = sum(1 for i, l in enumerate(labels_sample) if flagged[i] and l != 'normal')

print('Threat Hunting Results (kNN outlier scoring):')
print(f'  Total events sampled: {len(X_sample)}')
print(f'  Attack events in sample: {n_attacks}')
print(f'  Flagged as suspicious: {flagged.sum()}')
print(f'  Attack events flagged: {flagged_attacks}/{n_attacks}')
print(f'  Attack detection rate: {flagged_attacks/n_attacks:.3f}')
print(f'  Analyst queue: {flagged.sum()} events (vs {n_attacks} actual attacks)')

# Breakdown by attack type in flagged set
print('\nFlagged events by type:')
from collections import Counter
flagged_types = Counter(l for i, l in enumerate(labels_sample) if flagged[i])
for k, v in sorted(flagged_types.items()):
    print(f'  {k}: {v}')
